# A1.5 · Blast radius as a design metric

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.4 · Designing the agent control plane](https://spbreed.github.io/cyber-commons/lessons/A1.4.html)**.

| | |
|---|---|
| Open-source tooling | OpenFGA, SPIRE |
| Open-weight models | Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


"Reduce the blast radius" is advice. Advice does not survive a roadmap
discussion, because it cannot be traded off against a delivery date.

A **number** survives. It moves when you change the design, you can put it in a
review, and — most usefully — it can go into CI and fail a build.

The metric used throughout this curriculum is deliberately crude:

    blast radius = Σ over state-changing tools of
                     scope_weight × (2 if irreversible) × (0 if gated)

The absolute value means nothing. The **ratio between two designs** means a
great deal, and that is all a design metric has to do. Anyone who demands a
calibrated number before measuring anything ends up measuring nothing.

## 2 · Demo — four ways to build the same capability

The requirement: an agent that can triage security findings, fix simple ones, and deploy the fix. Four architectures, all of which deliver it.

In [ ]:
from dataclasses import dataclass

SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20, "internet": 50}

@dataclass(frozen=True)
class Tool:
    name: str; writes: bool = False; reversible: bool = True; scope: str = "self"

def blast(tools, gated=frozenset()):
    total = 0
    for t in tools:
        if not t.writes or t.name in gated:
            continue
        total += SCOPE_WEIGHT[t.scope] * (1 if t.reversible else 2)
    return total

READ   = [Tool("read_findings"), Tool("read_source")]
FIX    = [Tool("write_file", writes=True, scope="project"),
          Tool("open_pr",    writes=True, scope="project")]
SHIP   = [Tool("merge_pr", writes=True, scope="project", reversible=False),
          Tool("deploy",   writes=True, scope="org",     reversible=False)]

designs = {
    "A · one agent, everything":        (READ + FIX + SHIP, set()),
    "B · one agent, gate the shipping": (READ + FIX + SHIP, {"merge_pr", "deploy"}),
    "C · two agents (fixer / shipper)": (READ + FIX,        set()),
    "D · two agents + gated shipper":   (READ + FIX,        set()),
}
for name, (tools, gated) in designs.items():
    print(f"{name:36s} blast = {blast(tools, gated):3d}")

print("\nDesign D's shipper agent, measured separately:")
print(f"{'    shipper (gated)':36s} blast = {blast(SHIP, {'merge_pr','deploy'}):3d}")
print(f"{'    shipper (ungated)':36s} blast = {blast(SHIP):3d}   ← the honest number")

## 3 · Where it breaks — the number can lie

Design C looks best: blast 6. But it achieved that by *moving* the dangerous tools to another agent, not by removing them. If you measure each agent separately and report the lowest, you have optimised the metric rather than the risk. This is Goodhart's law arriving on schedule.

The metric is only honest when it is computed **over the whole system**, including every agent that can be reached from the first one.

In [ ]:
def system_blast(agents):
    """Sum across every agent in the system, not the one you are reviewing."""
    return sum(blast(tools, gated) for tools, gated in agents.values())

split_honest = {
    "fixer":   (READ + FIX, set()),
    "shipper": (SHIP,       set()),          # someone still runs this
}
split_gated = {
    "fixer":   (READ + FIX, set()),
    "shipper": (SHIP,       {"merge_pr", "deploy"}),
}
mono = {"one-agent": (READ + FIX + SHIP, set())}

for name, agents in (("monolith", mono), ("split, shipper ungated", split_honest),
                     ("split, shipper gated", split_gated)):
    print(f"{name:26s} system blast = {system_blast(agents):3d}")
print("\nSplitting alone bought nothing. Splitting AND gating bought everything.")
print("Reporting only the fixer's number would have hidden that.")

## 4 · The control — put it in CI

A metric nobody computes is a metric nobody has. The version that works is a budget, enforced by the build.

In [ ]:
BUDGETS = {"L1": 0, "L2": 0, "L2.5": 20, "L3": 60}

def check_budget(system, rung):
    total = system_blast(system)
    budget = BUDGETS[rung]
    ok = total <= budget
    return ok, (f"system blast {total} {'≤' if ok else '>'} budget {budget} "
                f"for rung {rung}")

for name, system, rung in [
    ("split + gated shipper", split_gated,  "L2.5"),
    ("split, ungated shipper", split_honest, "L2.5"),
    ("monolith",              mono,         "L2.5"),
]:
    ok, msg = check_budget(system, rung)
    print(f"{'PASS' if ok else 'FAIL'}  {name:26s} {msg}")

ok, _ = check_budget(split_gated, "L2.5")
assert ok, "the intended design must pass its own budget"
print("\nWired into CI, adding a tool now fails the build unless someone either")
print("gates it or raises the budget deliberately — which is a decision with a name on it.")

## 6 · The review, written down as a skill

A number computed once is a fact about today. The skill below is the same computation as a procedure someone else can run, and its contract requires `blast_radius.inputs` beside the score.

That requirement is the point: a metric nobody can decompose is a metric nobody can challenge, and an unchallengeable metric quietly stops being used.

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/architecture/blast-radius-review/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: blast-radius-review
description: >-
  Compute what an agent can reach and damage in a single run, and decide the
  autonomy level its blast radius can support. Use when reviewing an agent
  design or deployment, deciding whether an action needs human approval, sizing
  a sandbox, or answering how bad it would be if an agent were fully
  compromised.
allowed-tools: Read, Grep, Glob
---

# Blast radius as a design metric

Blast radius is not an adjective. It is the set of resources an agent can
change before anyone can stop it, and it is **computed** from three inputs:

```
blast_radius = reachable_resources × action_irreversibility × time_to_human_stop
```

Treating it as a number is what lets it be a design constraint instead of a
discussion.

## When to use this

At design review, before raising an agent's autonomy, and after any change that
adds a tool, a credential, or a scheduled trigger.

## Procedure

**1 — Enumerate reachable resources.** For each tool the agent can call, list
what it can touch with attacker-chosen arguments — not what it touches in the
happy path. A `Bash` tool with unrestricted arguments reaches everything the
process can reach; record it that way rather than as one row.

**2 — Grade irreversibility.** Per action:

| Grade | Meaning | Example |
|---|---|---|
| 0 | read-only | query, list |
| 1 | reversible with effort | write a file, open a PR |
| 2 | reversible only with a backup | delete a row, force-push |
| 3 | irreversible or externally visible | send an email, pay, publish, rotate a key |

Grade 3 actions are the whole reason approval gates exist. An agent whose
worst action is grade 0 does not need one.

**3 — Measure time-to-human-stop.** How long between the agent deciding and a
human being able to intervene? Interactive with a prompt is seconds. A
scheduled run at 03:00 with notifications off is hours. This term dominates the
product more often than people expect, and it is usually the cheapest to fix.

**4 — Place it on the autonomy ladder.**

| Level | Meaning | Requires |
|---|---|---|
| L1 | suggests; human executes | nothing |
| L2 | acts within a bounded sandbox | reversible actions only |
| L2.5 | acts, but grade-3 actions need approval | a working approval path |
| L3 | acts unattended | demonstrated containment + audit + stop authority |

An agent at L3 whose grade-3 actions are unbounded is misclassified, not brave.

**5 — Find the cheapest reduction.** Usually one of: remove a credential from
the environment, split one broad tool into two narrow ones, add a choke point
in front of the irreversible action, or shorten time-to-stop with a
notification. Recommend the one with the best radius reduction per unit of
friction, and say what it costs.

## Output contract

```json
{
  "resources": [{"tool": "str", "reachable": ["str"], "unbounded": false}],
  "actions": [{"action": "str", "irreversibility": 0, "why": "str"}],
  "time_to_human_stop_seconds": 0,
  "blast_radius": {"score": 0, "inputs": {"resources": 0, "max_irreversibility": 0, "seconds": 0}},
  "autonomy": {"current": "L1|L2|L2.5|L3", "supported": "L1|L2|L2.5|L3", "mismatch": false},
  "reductions": [{"change": "str", "new_score": 0, "friction": "low|medium|high"}]
}
```

Show `inputs`. A blast-radius score without its terms cannot be challenged, and
an unchallengeable metric stops being used.

## Failure modes

- **Counting the happy path.** Enumerate with attacker-chosen arguments.
- **Ignoring time-to-stop** because it is not about permissions. It is the term
  that separates an incident from a near miss.
- **Raising autonomy because the agent has been reliable.** Reliability is not
  containment; it is the absence of an adversary so far.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
contract = contract_of(body)
TOOLS = READ + FIX + SHIP

def irreversibility(t):
    if not t.writes:      return 0                      # read-only
    if t.reversible:      return 1 if t.scope in ("self", "project") else 2
    return 3                                            # irreversible

# An agent that runs unattended overnight has a much longer path to a human
# than one that prompts. This term is usually the cheapest of the three to fix.
TIME_TO_STOP = 8 * 60 * 60

review = {
 "resources": [{"tool": t.name, "reachable": [t.scope],
                "unbounded": t.scope in ("org", "internet")} for t in TOOLS],
 "actions": [{"action": t.name, "irreversibility": irreversibility(t),
              "why": ("read-only" if not t.writes else
                      f"writes at {t.scope} scope, "
                      f"{'reversible' if t.reversible else 'irreversible'}")}
             for t in TOOLS],
 "time_to_human_stop_seconds": TIME_TO_STOP,
 "blast_radius": {"score": blast(TOOLS),
                  "inputs": {"resources": len(TOOLS),
                             "max_irreversibility": max(irreversibility(t) for t in TOOLS),
                             "seconds": TIME_TO_STOP}},
 "autonomy": {"current": "L3", "supported": "L2.5", "mismatch": True},
 "reductions": [{"change": f"gate {t.name} behind approval",
                 "new_score": blast(TOOLS, gated=frozenset({t.name})),
                 "friction": "low"}
                for t in TOOLS if irreversibility(t) == 3],
}
problems = check(review, contract)
print(f"conformance: {len(problems)} problem(s)")
for p in problems: print("   ", p)
assert not problems, problems

print(f"\nblast radius {review['blast_radius']['score']} from "
      f"{review['blast_radius']['inputs']['resources']} tools, worst action grade "
      f"{review['blast_radius']['inputs']['max_irreversibility']}")
print(f"autonomy claimed {review['autonomy']['current']}, "
      f"supported {review['autonomy']['supported']} -> mismatch "
      f"{review['autonomy']['mismatch']}")
print("\ncheapest reductions:")
for r in sorted(review["reductions"], key=lambda r: (r["new_score"], r["change"])):
    print(f"   {r['change']:38s} {review['blast_radius']['score']} -> {r['new_score']}")
print()
print("The mismatch is the finding. Grade-3 actions are why approval gates")
print("exist, and an agent running unattended with one is at L3 by deployment")
print("and L2.5 by design - misclassified, not brave.")
assert review["autonomy"]["mismatch"]
assert any(a["irreversibility"] == 3 for a in review["actions"])

## What you just proved

Design A scores 92, B scores 6, C scores 6. Measured across the whole system, the ungated split still scores 92 while the gated split scores 6 — showing that splitting alone bought nothing. The budget check passes only the gated design.

## Your turn

Compute the system blast radius for your largest agent deployment, counting every agent it can invoke. Then pick a budget and see how many of your current designs would fail it. Set the budget at today's number and ratchet down; a budget nothing passes is ignored by lunchtime.

---

**Next → [A1.6 · Multi-agent topology](https://spbreed.github.io/cyber-commons/lessons/A1.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*